# @before_agent 与 @after_agent

Agent 级别的钩子，分别在 Agent 执行前和执行后各执行一次。适合做初始化、预处理、后处理。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("导入完成")

导入完成


## before_agent——Agent 开始前的准备工作

### 场景 1：输入预处理

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

@before_agent
def preprocess_input(state, runtime):
    """在 Agent 开始前处理用户输入"""
    messages = state.get("messages", [])
    if not messages:
        return None
    return None

@tool
def search_course(keyword: str) -> str:
    """在菜鸟教程搜索课程"""
    courses = {"python": "Python3 基础教程（免费，30章）", "html": "HTML 基础教程（免费，25章）"}
    return courses.get(keyword.lower(), f"未找到 {keyword}")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, tools=[search_course], middleware=[preprocess_input], system_prompt="你是菜鸟教程的课程顾问。")

result = agent.invoke({"messages": [HumanMessage(content="Python 课程")]})
print(f"回复: {result['messages'][-1].content}")

print("预处理中间件已创建")

回复: ## 🐍 Python 课程介绍

在菜鸟教程中，我们提供了一门 **Python3 基础教程**，以下是详细信息：

---

### 📘 Python3 基础教程

| 项目 | 内容 |
|------|------|
| **课程性质** | ✅ **免费课程** |
| **章节数量** | 📚 **共 30 章** |
| **适合人群** | 零基础初学者、想系统学习 Python 的开发者 |
| **内容涵盖** | Python 基础语法、数据类型、流程控制、函数、模块、文件操作、面向对象等核心知识点 |

---

### 🎯 课程特色
- **由浅入深**：从环境搭建到高级特性，系统化学习路径
- **实操案例**：配有大量代码示例，理论与实践结合
- **在线运行**：支持在线编写和运行 Python 代码
- **持续更新**：紧跟 Python 最新版本，内容与时俱进

---

### 💡 适合这样的你
- 🔹 编程新手，想入门 Python
- 🔹 在校学生，需要学习 Python 课程
- 🔹 数据分析、Web 开发、自动化脚本感兴趣的同学
- 🔹 想打好 Python 基础的开发者

---

**现在就可以免费学习！** 直接访问菜鸟教程官网，搜索 **"Python3"** 即可开始你的 Python 学习之旅 🚀

请问您想进一步了解某个具体章节的内容，或者有其他关于课程的疑问吗？😊
预处理中间件已创建


### 场景 2：访问控制——权限检查

In [3]:
from langchain.agents.middleware import before_agent

@before_agent
def access_control(state, runtime):
    """检查用户权限"""
    context = runtime.context
    if context is None:
        return None
    user_role = context.get("user_role", "guest")
    if user_role == "guest":
        messages = state.get("messages", [])
        if messages:
            last_content = str(messages[-1].content)
            restricted = ["删除", "管理", "配置", "admin"]
            if any(kw in last_content for kw in restricted):
                return {"jump_to": "end", "messages": [HumanMessage(content="权限不足，请登录后重试。")]}
    return None

print("权限检查中间件已定义")

权限检查中间件已定义


## after_agent——Agent 完成后的处理

### 场景 3：统计分析

In [4]:
from langchain.agents.middleware import after_agent
from langchain.tools import tool

@after_agent
def conversation_stats(state, runtime):
    """统计对话信息"""
    messages = state.get("messages", [])
    model_calls = sum(1 for m in messages if m.type == "ai")
    tool_calls = sum(len(m.tool_calls) for m in messages if hasattr(m, 'tool_calls') and m.tool_calls)
    runtime.stream_writer({"type": "stats", "model_calls": model_calls, "tool_calls": tool_calls, "total": len(messages)})
    return None

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
@tool
def search_course(keyword: str) -> str:
    """搜索课程"""
    return f"找到 {keyword} 课程"

agent = create_agent(model=model, tools=[search_course], middleware=[conversation_stats], system_prompt="你是课程顾问。")

for mode, chunk in agent.stream({"messages": [HumanMessage(content="查一下 Python")]}, stream_mode=["updates", "custom"]):
    if mode == "custom" and chunk.get("type") == "stats":
        print(f"统计: {chunk}")

print("统计中间件已创建")

统计: {'type': 'stats', 'model_calls': 2, 'tool_calls': 1, 'total': 4}
统计中间件已创建


### 场景 4：格式化输出

In [5]:
from langchain.agents.middleware import after_agent
from langchain.messages import AIMessage

@after_agent
def format_output(state, runtime):
    """追加格式化总结"""
    messages = state.get("messages", [])
    if not messages:
        return None
    last_ai = None
    for msg in reversed(messages):
        if msg.type == "ai" and msg.content:
            last_ai = msg
            break
    if last_ai:
        tool_msgs = [m for m in messages if m.type == "tool"]
        footer = f"\n\n---\n> 共 {len(messages)} 条消息，{len(tool_msgs)} 次工具调用。"
        return {"messages": [AIMessage(content=last_ai.content + footer)]}
    return None

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
@tool
def search_course(keyword: str) -> str:
    """搜索课程"""
    return f"找到 {keyword} 相关课程"

agent = create_agent(model=model, tools=[search_course], middleware=[format_output], system_prompt="你是课程顾问。")

result = agent.invoke({"messages": [HumanMessage(content="Python 有什么课程？")]})
print(result["messages"][-1].content)

print("格式化输出中间件已创建")

好的，我们目前有与 **Python** 相关的课程资源！不过为了给您推荐最合适的课程，我想先了解一下您的具体情况：

---

## 📌 请您告诉我以下信息：

1. **您目前的编程基础如何？**
   - 🔹 零基础，完全没接触过编程
   - 🔹 有一些编程基础（如学过其他语言）
   - 🔹 有一定 Python 基础，想进阶学习

2. **您的学习目标是什么？**
   - 🖥️ 数据分析 / 人工智能
   - 🌐 Web 开发（Django/Flask）
   - 🤖 自动化办公 / 爬虫
   - 📚 考取 Python 认证
   - 🎯 其他方向

3. **您希望的课程形式？**
   - 🎬 录播视频课程
   - 🏫 直播互动课程
   - 👨‍🏫 一对一辅导

---

这样我可以精准为您匹配最适合的 Python 课程！您方便说说您的需求吗？😊

---
> 共 6 条消息，2 次工具调用。
格式化输出中间件已创建


## 四个钩子的完整协作示例

In [6]:
from langchain.agents.middleware import before_agent, after_agent, before_model, after_model
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

@before_agent
def init_session(state, runtime):
    print(">>> 会话开始")
    return None

@before_model
def pre_check(state, runtime):
    print(f" [model前] 消息数: {len(state.get('messages',[]))}")
    return None

@after_model
def post_check(state, runtime):
    last = state["messages"][-1] if state.get("messages") else None
    if last and hasattr(last, 'tool_calls') and last.tool_calls:
        print(f" [model后] 需要工具调用")
    return None

@after_agent
def finish_session(state, runtime):
    print(f"<<< 会话结束，共 {len(state.get('messages',[]))} 条")
    return None

@tool
def get_weather(city: str) -> str:
    """查询天气"""
    return f"{city}: 晴"

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, tools=[get_weather],
    middleware=[init_session, pre_check, post_check, finish_session],
    system_prompt="你是助手。",
)

result = agent.invoke({"messages": [HumanMessage(content="杭州天气？")]})
print(f"\n回复: {result['messages'][-1].content}")

print("完整协作 Agent 已创建")

>>> 会话开始
 [model前] 消息数: 1
 [model后] 需要工具调用
 [model前] 消息数: 3
<<< 会话结束，共 4 条

回复: 杭州今天天气是 **☀️ 晴**，是个好天气！祝你心情愉快~ 😊
完整协作 Agent 已创建


## Middleware 钩子总结

| 钩子 | 执行次数 | 关键能力 |
| --- | --- | --- |
| before_agent | 1次 | 权限检查、初始化、可 jump_to 提前终止 |
| before_model | 每次循环 | 消息裁剪、内容过滤、上下文注入 |
| wrap_model_call | 每次循环 | 重试、降级、缓存、完全控制模型 |
| after_model | 每次循环 | 响应审核、内容追加、日志 |
| wrap_tool_call | 每次工具调用 | 重试、缓存、参数改写 |
| after_agent | 1次 | 格式化输出、统计分析、清理 |

**术语：** before_agent、after_agent、Agent 级别钩子、会话生命周期